# 1. Install Dependencies

In [16]:
!pip install -U pip
!pip install TTS
!pip install torchcodec

  Using cached torchcodec-0.9.1-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (11 kB)
Using cached torchcodec-0.9.1-cp310-cp310-manylinux_2_28_x86_64.whl (2.0 MB)


# 2. Clone Repository & Setup

In [ ]:
!git clone https://VanModers:@github.com/VanModers/oostfraeisk_text_to_speech

In [ ]:
%cd oostfraeisk_text_to_speech

In [ ]:
!git pull

# 3. A100 GPU Optimizations

In [2]:
import torch

# Enable TF32 for faster matmul on A100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"TF32 enabled: {torch.backends.cuda.matmul.allow_tf32}")

GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
VRAM: 102.0 GB
TF32 enabled: True


# 4. Download Pretrained German VITS Model

In [3]:
from TTS.utils.manage import ModelManager

manager = ModelManager()
model_path, config_path, _ = manager.download_model("tts_models/de/thorsten/vits")
print(f"Model: {model_path}")
print(f"Config: {config_path}")

 > tts_models/de/thorsten/vits is already downloaded.
Model: /home/tidospecht/.local/share/tts/tts_models--de--thorsten--vits/model_file.pth
Config: /home/tidospecht/.local/share/tts/tts_models--de--thorsten--vits/config.json


# 5. Define East Frisian Character Set

For grapheme-only training, we need to define all characters that appear in the dataset.

In [8]:
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.configs.shared_configs import CharactersConfig

# All characters that appear in East Frisian text
# Standard letters
letters_lower = "abcdefghijklmnopqrstuvwxyz"
letters_upper = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

# East Frisian special characters (including circumflex vowels)
# â ê î ô û = extra-long vowels
# ä ö ü = umlauts
# ó = open o sound
# ğ = velar fricative (like Dutch g)
east_frisian_special = "ÂÊÎÔÛâêîôûÄÖÜäöüÓóĞğß"

# Numbers (in case they appear in text)
numbers = "0123456789"

# All grapheme characters (no punctuation here)
all_characters = letters_lower + letters_upper + east_frisian_special + numbers

# Punctuation marks
punctuations = "!\"'(),-.:;? \n"

print(f"Character set ({len(all_characters)} chars): {all_characters}")
print(f"Punctuations: {repr(punctuations)}")

Character set (83 chars): abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZÂÊÎÔÛâêîôûÄÖÜäöüÓóĞğß0123456789
Punctuations: '!"\'(),-.:;? \n'


# 6. Configure Model for Grapheme-Only Training

In [9]:
# Create custom character config for East Frisian graphemes
characters_config = CharactersConfig(
    characters_class="TTS.tts.utils.text.characters.Graphemes",
    characters=all_characters,
    punctuations=punctuations,
    pad="<PAD>",
    eos="<EOS>",
    bos="<BOS>",
    blank="<BLNK>",
)

# Load pretrained config as base
config = VitsConfig()
config.load_json(config_path)

# === CRITICAL: Use graphemes, not phonemes ===
config.use_phonemes = False
config.characters = characters_config

# Dataset settings
config.datasets[0].formatter = "ljspeech"
config.datasets[0].path = "data/oostfraeisk"
config.datasets[0].meta_file_train = "metadata.csv"

# Training settings
config.output_path = "tts_train_dir_grapheme"
config.epochs = 100
config.run_eval = True
config.test_delay_epochs = -1

# === A100 OPTIMIZED SETTINGS ===
config.batch_size = 8 
config.eval_batch_size = 16
config.num_loader_workers = 8 
config.num_eval_loader_workers = 4

config.save_step = 500

# Logging
config.print_step = 100
config.log_model_step = 1000

# Mixed precision & cuDNN
config.mixed_precision = True
config.cudnn_benchmark = True

print("=" * 50)
print("GRAPHEME-ONLY TRAINING CONFIG")
print("=" * 50)
print(f"use_phonemes: {config.use_phonemes}")
print(f"characters_class: {config.characters.characters_class}")
print(f"Batch size: {config.batch_size}")
print(f"Learning rate: {config.lr_gen}")
print(f"Output path: {config.output_path}")

GRAPHEME-ONLY TRAINING CONFIG
use_phonemes: False
characters_class: TTS.tts.utils.text.characters.Graphemes
Batch size: 8
Learning rate: 0.0002
Output path: tts_train_dir_grapheme


# 7. Verify Dataset

Check that all characters in the dataset are covered by our character set.

In [10]:
from pathlib import Path

metadata_path = Path("data/oostfraeisk/metadata.csv")

# Read all text from metadata
all_text = ""
with open(metadata_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('|')
        if len(parts) >= 2:
            all_text += parts[1] + " "

# Find unique characters in dataset
dataset_chars = set(all_text)
config_chars = set(all_characters + punctuations)

# Check for missing characters
missing = dataset_chars - config_chars
if missing:
    print(f"⚠️  Missing characters in config: {missing}")
    print(f"   Add these to 'all_characters' or 'punctuations'")
else:
    print(f"✓ All {len(dataset_chars)} unique characters are covered!")

# Show character frequency
from collections import Counter
char_freq = Counter(all_text)
print(f"\nMost common characters:")
for char, count in char_freq.most_common(20):
    print(f"  '{char}': {count}")

⚠️  Missing characters in config: {'–', '\xa0', '¬', '/'}
   Add these to 'all_characters' or 'punctuations'

Most common characters:
  ' ': 12889
  'e': 6045
  'n': 5762
  'i': 4416
  't': 4151
  'a': 3419
  'r': 3266
  's': 3058
  'd': 2886
  'o': 2281
  'u': 2280
  'l': 2121
  'k': 1985
  'm': 1659
  'ó': 1529
  'ä': 1438
  ''': 1365
  'h': 1226
  'f': 1193
  'ö': 1090


# 8. Initialize Model and Tokenizer

In [11]:
from TTS.tts.models.vits import Vits
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer

# Initialize audio processor and tokenizer
ap = AudioProcessor.init_from_config(config)
tokenizer, config = TTSTokenizer.init_from_config(config)

print(f"Tokenizer type: {type(tokenizer).__name__}")
print(f"Vocab size: {tokenizer.characters.num_chars}")
print(f"Characters: {tokenizer.characters.characters}")

 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
Tokenizer type: TTSTokenizer
Vocab size: 100
Characters: abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZÂÊÎÔÛâêîôûÄÖÜäöüÓóĞğß0123456789


In [12]:
# Test tokenization with East Frisian text
test_sentences = [
    "Moin, woo gaajt 't dii?",
    "Däi süen skint up us land.",
    "Wii prootent Oostfräisk.",
    "Hest duu däi süen fandóóeğ al säin?",
]

print("Tokenization test:")
print("=" * 50)
for sent in test_sentences:
    try:
        ids = tokenizer.text_to_ids(sent)
        back = tokenizer.ids_to_text(ids)
        print(f"Input:  {sent}")
        print(f"IDs:    {ids[:20]}..." if len(ids) > 20 else f"IDs:    {ids}")
        print(f"Back:   {back}")
        print()
    except Exception as e:
        print(f"ERROR with '{sent}': {e}")
        print()

Tokenization test:
Input:  Moin, woo gaajt 't dii?
IDs:    [3, 52, 3, 54, 3, 48, 3, 53, 3, 92, 3, 98, 3, 62, 3, 54, 3, 54, 3, 98]...
Back:   <BLNK>m<BLNK>o<BLNK>i<BLNK>n<BLNK>,<BLNK> <BLNK>w<BLNK>o<BLNK>o<BLNK> <BLNK>g<BLNK>a<BLNK>a<BLNK>j<BLNK>t<BLNK> <BLNK>'<BLNK>t<BLNK> <BLNK>d<BLNK>i<BLNK>i<BLNK>?<BLNK>

Input:  Däi süen skint up us land.
IDs:    [3, 43, 3, 77, 3, 48, 3, 98, 3, 58, 3, 84, 3, 44, 3, 53, 3, 98, 3, 58]...
Back:   <BLNK>d<BLNK>ä<BLNK>i<BLNK> <BLNK>s<BLNK>ü<BLNK>e<BLNK>n<BLNK> <BLNK>s<BLNK>k<BLNK>i<BLNK>n<BLNK>t<BLNK> <BLNK>u<BLNK>p<BLNK> <BLNK>u<BLNK>s<BLNK> <BLNK>l<BLNK>a<BLNK>n<BLNK>d<BLNK>.<BLNK>

Input:  Wii prootent Oostfräisk.
IDs:    [3, 62, 3, 48, 3, 48, 3, 98, 3, 55, 3, 57, 3, 54, 3, 54, 3, 59, 3, 44]...
Back:   <BLNK>w<BLNK>i<BLNK>i<BLNK> <BLNK>p<BLNK>r<BLNK>o<BLNK>o<BLNK>t<BLNK>e<BLNK>n<BLNK>t<BLNK> <BLNK>o<BLNK>o<BLNK>s<BLNK>t<BLNK>f<BLNK>r<BLNK>ä<BLNK>i<BLNK>s<BLNK>k<BLNK>.<BLNK>

Input:  Hest duu däi süen fandóóeğ al säin?
IDs:    [3, 47, 3, 44, 3, 58, 3,

# 9. Load Pretrained Model

We load the German VITS model but with `strict=False` since the embedding layer size will differ (graphemes vs phonemes).

In [14]:
import torch

model = Vits.init_from_config(config)

# Load checkpoint manually
checkpoint = torch.load(model_path, map_location="cpu")
state_dict = checkpoint["model"]

# Remove the mismatched embedding layer from state_dict
if "text_encoder.emb.weight" in state_dict:
    del state_dict["text_encoder.emb.weight"]

# Load remaining weights
model.load_state_dict(state_dict, strict=False)

print(f"Model loaded successfully!")

 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
Model loaded successfully!


# 10. Train the Model

In [17]:
from TTS.tts.datasets import load_tts_samples
from trainer import Trainer, TrainerArgs

# Load dataset
train_samples, eval_samples = load_tts_samples(
    config.datasets[0],
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)

print(f"Train samples: {len(train_samples)}")
print(f"Eval samples: {len(eval_samples)}")

# Initialize trainer
trainer = Trainer(
    TrainerArgs(),
    config,
    config.output_path,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

# Start training
trainer.fit()

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: fp16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 24
 | > Num. of Torch Threads: 24
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: True
 | > Torch TF32 MatMul: True
 > Start Tensorboard: tensorboard --logdir=tts_train_dir_grapheme/vits-0.6.1-Thorsten-DE-January-08-2026_11+52AM-1908f1c
/home/tidospecht/Downloads/oostfraeisk_text_to_speech/.venv/lib/python3.10/site-packages/trainer/trainer.py:552: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()

 > Model has 83053228 parameters


 | > Found 820 files in /home/tidospecht/Downloads/oostfraeisk_text_to_speech/data/oostfraeisk
Train samples: 812
Eval samples: 8



 > EPOCH: 0/100
 --> tts_train_dir_grapheme/vits-0.6.1-Thorsten-DE-January-08-2026_11+52AM-1908f1c

 > TRAINING (2026-01-08 11:52:35) 




> DataLoader initialization
| > Tokenizer:
	| > add_blank: True
	| > use_eos_bos: False
	| > use_phonemes: False
| > Number of instances : 812
 | > Preprocessing samples
 | > Max text length: 178
 | > Min text length: 4
 | > Avg text length: 89.42487684729063
 | 
 | > Max audio length: 388118.0
 | > Min audio length: 8214.0
 | > Avg audio length: 146245.31527093597
 | > Num. instances discarded samples: 0
 | > Batch group size: 40.


/home/tidospecht/Downloads/oostfraeisk_text_to_speech/.venv/lib/python3.10/site-packages/torch/functional.py:681: UserWarning: stft with return_complex=False is deprecated. In a future pytorch release, stft will return complex tensors for all inputs, and return_complex=False will raise an error.
Note: you can still call torch.view_as_real on the complex output to recover the old return format. (Triggered internally at /pytorch/aten/src/ATen/native/SpectralOps.cpp:875.)
  return _VF.stft(  # type: ignore[attr-defined]
/home/tidospecht/Downloads/oostfraeisk_text_to_speech/.venv/lib/python3.10/site-packages/TTS/tts/models/vits.py:1273: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):  # use float32 for the criterion
/home/tidospecht/Downloads/oostfraeisk_text_to_speech/.venv/lib/python3.10/site-packages/TTS/tts/models/vits.py:1284: FutureWarning: `torch.cuda.amp.autocast(args...)` is 

...


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.31658172607421875 (-0.01663517951965332)
     | > avg_loss_disc: 2.6517722606658936 (+0.0759279727935791)
     | > avg_loss_disc_real_0: 0.14164575934410095 (-0.018078893423080444)
     | > avg_loss_disc_real_1: 0.2630988657474518 (+0.04937402904033661)
     | > avg_loss_disc_real_2: 0.2661522924900055 (+0.04572710394859314)
     | > avg_loss_disc_real_3: 0.2309097796678543 (+0.0018277168273925781)
     | > avg_loss_disc_real_4: 0.3176026940345764 (+0.15206657350063324)
     | > avg_loss_disc_real_5: 0.2283427119255066 (-0.0012488961219787598)
     | > avg_loss_0: 2.6517722606658936 (+0.0759279727935791)
     | > avg_loss_gen: 2.171722173690796 (+0.13637590408325195)
     | > avg_loss_kl: 1.8660684823989868 (-0.1433166265487671)
     | > avg_loss_feat: 5.729723930358887 (-2.6082515716552734)
     | > avg_loss_mel: 19.78306770324707 (+0.24928665161132812)
     | > avg_loss_duration: 1.8124152421951294 (+0.0005983114242553711)
     | > 

gräel süen t' möyet¬. nóó 'n setje sağ ik in d' fêert 'n blaau lücht
 [!] Character '¬' not found in the vocabulary. Discarding it.
tom see teegen marió, häi löövde, dat 'n niilpeerd 'n loopgaauiğkaid fan 30 km/h berekken kun.
 [!] Character '/' not found in the vocabulary. Discarding it.
däi wustjes bünt mii bii 't kooken in 't häiet wóóter spiitelkerwîis busten. – móókt niks, tom! däi smóókent trotsdeem!
 [!] Character '–' not found in the vocabulary. Discarding it.



 > EVALUATION 

   --> STEP: 0
     | > loss_disc: 2.577557325363159  (2.577557325363159)
     | > loss_disc_real_0: 0.18459153175354004  (0.18459153175354004)
     | > loss_disc_real_1: 0.22080276906490326  (0.22080276906490326)
     | > loss_disc_real_2: 0.25889691710472107  (0.25889691710472107)
     | > loss_disc_real_3: 0.28881871700286865  (0.28881871700286865)
     | > loss_disc_real_4: 0.23554877936840057  (0.23554877936840057)
     | > loss_disc_real_5: 0.2863122224807739  (0.2863122224807739)
     | > loss_0: 2.577557325363159  (2.577557325363159)
     | > loss_gen: 2.3617799282073975  (2.3617799282073975)
     | > loss_kl: 2.408350944519043  (2.408350944519043)
     | > loss_feat: 7.902956962585449  (7.902956962585449)
     | > loss_mel: 19.723201751708984  (19.723201751708984)
     | > loss_duration: 1.8349609375  (1.8349609375)
     | > loss_1: 34.23125076293945  (34.23125076293945)



 | > Synthesizing test sentences.



  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.330902099609375 (+0.01432037353515625)
     | > avg_loss_disc: 2.577557325363159 (-0.07421493530273438)
     | > avg_loss_disc_real_0: 0.18459153175354004 (+0.04294577240943909)
     | > avg_loss_disc_real_1: 0.22080276906490326 (-0.04229609668254852)
     | > avg_loss_disc_real_2: 0.25889691710472107 (-0.007255375385284424)
     | > avg_loss_disc_real_3: 0.28881871700286865 (+0.05790893733501434)
     | > avg_loss_disc_real_4: 0.23554877936840057 (-0.08205391466617584)
     | > avg_loss_disc_real_5: 0.2863122224807739 (+0.057969510555267334)
     | > avg_loss_0: 2.577557325363159 (-0.07421493530273438)
     | > avg_loss_gen: 2.3617799282073975 (+0.19005775451660156)
     | > avg_loss_kl: 2.408350944519043 (+0.5422824621200562)
     | > avg_loss_feat: 7.902956962585449 (+2.1732330322265625)
     | > avg_loss_mel: 19.723201751708984 (-0.05986595153808594)
     | > avg_loss_duration: 1.8349609375 (+0.022545695304870605)
     | > avg_los

# 11. Find Best Checkpoint

In [18]:
import glob

output_path = "tts_train_dir_grapheme"
ckpts = sorted(glob.glob(output_path + "/*/*.pth"))
configs = sorted(glob.glob(output_path + "/*/*.json"))

print("Available checkpoints:")
for ckpt in ckpts[-5:]:
    print(f"  {ckpt}")

print(f"\nLatest config: {configs[-1] if configs else 'None'}")

Available checkpoints:
  tts_train_dir_grapheme/vits-0.6.1-Thorsten-DE-January-08-2026_11+52AM-1908f1c/checkpoint_10000.pth
  tts_train_dir_grapheme/vits-0.6.1-Thorsten-DE-January-08-2026_11+52AM-1908f1c/checkpoint_8000.pth
  tts_train_dir_grapheme/vits-0.6.1-Thorsten-DE-January-08-2026_11+52AM-1908f1c/checkpoint_8500.pth
  tts_train_dir_grapheme/vits-0.6.1-Thorsten-DE-January-08-2026_11+52AM-1908f1c/checkpoint_9000.pth
  tts_train_dir_grapheme/vits-0.6.1-Thorsten-DE-January-08-2026_11+52AM-1908f1c/checkpoint_9500.pth

Latest config: tts_train_dir_grapheme/vits-0.6.1-Thorsten-DE-January-08-2026_11+52AM-1908f1c/config.json


# 12. Test Inference

In [19]:
# Update these paths to your best checkpoint
best_checkpoint = ckpts[-1] if ckpts else None
best_config = configs[-1] if configs else None

if best_checkpoint:
    print(f"Using checkpoint: {best_checkpoint}")
    print(f"Using config: {best_config}")

Using checkpoint: tts_train_dir_grapheme/vits-0.6.1-Thorsten-DE-January-08-2026_11+52AM-1908f1c/checkpoint_9500.pth
Using config: tts_train_dir_grapheme/vits-0.6.1-Thorsten-DE-January-08-2026_11+52AM-1908f1c/config.json


In [20]:
# Generate speech
!tts --text "Moin, woo gaajt 't dii? Ik hoop, dat 't dii gaud gaajt." \
     --model_path {best_checkpoint} \
     --config_path {best_config} \
     --out_path out_grapheme.wav

 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Text: Moin, woo gaajt 't dii? Ik hoop, dat 't dii gaud gaajt.
 > Text splitted to sentences.
["Moin, woo gaajt 't dii?", "Ik hoop, dat 't dii gaud gaajt."]
 > Processing time: 4.193305969238281
 > Real-time factor: 0.86796332064531

In [22]:
# Generate speech
!tts --text "Denkent jii, dat ik disser sats gaud uutprooten dau?" \
     --model_path {best_checkpoint} \
     --config_path {best_config} \
     --out_path out_grapheme.wav

 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Text: Denkent jii, dat ik disser sats gaud uutprooten dau?
 > Text splitted to sentences.
['Denkent jii, dat ik disser sats gaud uutprooten dau?']
 > Processing time: 3.2112696170806885
 > Real-time factor: 0.7019081587691235
 > Sa

In [26]:
# Generate speech
!tts --text "Hest duu däi süen fandóóeğ al säin? Ik wäit näit, wor däi süen skint, man ik wäit dat däi minsken singent un up mii wachtent." \
     --model_path {best_checkpoint} \
     --config_path {best_config} \
     --out_path out_grapheme.wav

 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Text: Hest duu däi süen fandóóeğ al säin? Ik wäit näit, wor däi süen skint, man ik wäit dat däi minsken singent un up mii wachtent.
 > Text splitted to sentences.
['Hest duu däi süen fandóóeğ al säin?', 'Ik wäit näit, wor däi süen 

In [ ]:
# Generate speech
!tts --text "Däi oorsprungelk tóól fan däi Fräisen tüsken Laauwers un Wäiser was dat Olfräisk, genaauer Oosterlaauwersfräisk. Disser tóól wur ruuğweğ teegen 1500 döör dat Middelleeğdüütsk as skrifttóól ferdrungen, wiilst däi proottóól bii däi oostfräisk buren up 't êerst dat Oosterlaauwersfräisk bleev. In däi wiiderder ferloop fan däi tiid kwam dat dan tüsken 1500 un 1800 tau 'n recht sâacht tóólwessel, bii däi wal eerst däi börgers in d' steeden up dat Leeğdüütsk as proottóól wesseldent." \
     --model_path {best_checkpoint} \
     --config_path {best_config} \
     --out_path out_grapheme.wav

 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Text: Däi oorsprungelk tóól fan däi Fräisen tüsken Laauwers un Wäiser was dat Olfräisk, genaauer Oosterlaauwersfräisk. Disser tóól wur ruuğweğ teegen 1500 döör dat Middelleeğdüütsk as skrifttóól ferdrungen, wiilst däi proottóól bii

In [ ]:
# Generate speech
!tts --text "Liike is in 't säest joer in foelğ däi beläivst wichternóóm in Fräisland. Bii däi jungse fört Hidde föör 't fäärd móól in foelğ däi liest an. Disser dóóten kooment fan d' sosjóólferseekernsbank. 43 pupkes in us prowins wurdent Hidde nöymt. Liike kreeğ eksakt däi sülviğ tóól an nóómen." \
     --model_path {best_checkpoint} \
     --config_path {best_config} \
     --out_path out_grapheme.wav

 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Text: Liike is in 't säest joer in foelğ däi beläivst wichternóóm in Fräisland. Bii däi jungse fört Hidde föör 't fäärd móól in foelğ däi liest an. Disser dóóten kooment fan d' sosjóólferseekernsbank. 43 pupkes in us prowins wurden

In [43]:
import IPython
IPython.display.Audio("out_grapheme.wav")

# 13. Copy Best Model to model/ Directory

In [35]:
!mkdir -p model_grapheme

In [36]:
# Copy your best checkpoint (update the path as needed)
# Example: !cp tts_train_dir_grapheme/vits-.../checkpoint_10000.pth model_grapheme/model.pth
!cp {best_checkpoint} model_grapheme/model.pth
!cp {best_config} model_grapheme/config.json

# 14. Push to GitHub

In [37]:
!git lfs install
!git lfs track "*.pth"

Updated git hooks.
Git LFS initialized.
"*.pth" already supported


In [ ]:
!git config --global user.email "programmingstudios227@gmail.com"
!git config --global user.name "Tido Specht"
!git add -A
!git commit -m "Grapheme-only model trainäärt"

In [ ]:
!git push

# 15. Push to HuggingFace (Optional)

In [3]:
!pip install huggingface_hub
!pip install ipywidgets


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 21.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 31.2 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]




[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Clone your HuggingFace space
%cd ..
!git clone https://huggingface.co/spaces/VanModers114/East_Frisian_TTS

In [ ]:
# Copy model to HuggingFace repo
!cp oostfraeisk_text_to_speech/model_grapheme/model.pth East_Frisian_TTS/model.pth
!cp oostfraeisk_text_to_speech/model_grapheme/config.json East_Frisian_TTS/config.json

In [ ]:
%cd East_Frisian_TTS
!git lfs install
!git lfs track "*.pth"
!git add -A
!git commit -m "Grapheme-only model"
!git push